# Officina dei prompt

Un pomeriggio alla volta, con il prompt aperto qui accanto. Serve a cambiare una frase e guardare che cosa cambia, prima di spendere un'ora per sapere se ha mosso un numero.

Le note qui sono in italiano perché questo è il banco dell'autore; il codice e i commenti restano in inglese come nel resto del repository.

**Non c'è nessuna copia dei prompt in questo file.** I blocchi restano nei `.md` accanto ai moduli che li mandano, e le celle qui sotto fanno solo rileggere quei file al processo vivo. Una copia diverrebbe un secondo prompt da tenere allineato, che è esattamente il guasto che ha tenuto ferma la ricerca per due giorni a settembre.

Tre limiti dichiarati, perché contano più di quello che il banco offre:

1. **Un pomeriggio è n=1.** Il 3 settembre tutti e otto gli assi sono calati (media 3,49 → 2,93) dopo un giro di modifiche che a leggerle sembravano miglioramenti. Qui si decide *che cosa* provare; se abbia funzionato lo dice `python -m research.run`.
2. **Il giro non è veloce.** Una chiamata di devise misura 76–184 s (mediana ~140). Il banco non accorcia la chiamata: tiene lo stato fra una chiamata e l'altra e fa vedere quello che c'è in mezzo.
3. **Un blocco modificato qui è modificato nel repository.** Non c'è niente da ricopiare a mano dopo: si committa, si rigenera `docs/prompts/` e si misura.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import research.bench as bench  # noqa: E402 - the root has to be on the path first

print(sorted(bench.environment()))
print("impronta del prompt:", bench.reload_prompts())

# One row per call made in this session. Without it every comparison is a memory of how
# the last one read, and a memory of a document is not evidence about a prompt.
TRIED: list[dict[str, object]] = []

Le credenziali non stanno in `env.ps1` e non ci sono mai state: il router costruisce un `DefaultAzureCredential`, che trova la sessione `az` dentro l'`AZURE_CONFIG_DIR` nominato lì. Se la prima chiamata al modello fallisce per autenticazione, la cura è `az account get-access-token` in un terminale con quella stessa variabile — `az account show` legge la cache e riesce anche a token scaduto.

## 1. I blocchi che compongono il prompt

In [ ]:
for module in (
    "agents.experience_deviser",
    "shared.experience_prompt",
    "agents.experience_continuer",
):
    said = bench.blocks(module)
    print(f"{module}  —  {len(said)} blocchi, {sum(n for _, n in said)} caratteri")
    for name, size in said:
        print(f"    {name:22} {size:6}")
    print()

In [ ]:
BLOCK = ("agents.experience_deviser", "manner-tail")
print(bench.read(*BLOCK))

## 2. La casa

Tutto quello che decide un pomeriggio, in un dizionario solo. È lo stesso che `research/run.py` manda a `devise_experience`, e lo stesso che la cella dopo dà a `the_prompt` per leggere che cosa uscirebbe: fatto così, quello che si legge e quello che si manda non possono divergere.

In [ ]:
from research.households import Household, Memory, arguments

house = Household(
    name="banco",
    interests=("i treni", "le mappe vecchie"),
    avoid=("i ragni",),
    load="middle",
    ink="middle",
    span="middle",
    sheets=2,
    note="",
)
memory = Memory()  # nessuna storia: è il primo pomeriggio di questa casa
args = arguments(house, memory)

for name, value in args.items():
    shown = value if isinstance(value, str) else repr(value)
    print(f"{name:12} {shown[:110] or '—'}")

## 3. Il prompt intero, prima di pagarlo

Il mestiere — una `form` e un `move` da `methods/` — è filtrato su quello che la casa sa fare e poi scelto. Qui è **sorteggiato**, per vedere il prompt; nella chiamata vera `devise_experience` lo fa scegliere al modello con una chiamata a parte, e ripiega sul sorteggio se quella non riesce. Quale sia finito nel pomeriggio lo dice `built_from` più sotto.

In [ ]:
from agents.experience_deviser import the_prompt
from shared.methods import draw, load, runnable

here = runnable(load(), capabilities=args["capabilities"])
form, move = draw(here)
print(f"{len(here)} metodi eseguibili in questa casa · {form.method_id} + {move.method_id}\n")

prompt = the_prompt(**args, form=form, move=move)
print(f"{len(prompt)} caratteri in tutto\n")
print(prompt[-3000:])

## 4. La chiamata

Il percorso vero e per intero: scelta del mestiere, chiamata, lettura del formato, una riparazione se serve, i controlli, il filtro di sicurezza. Il logging è acceso perché le riparazioni sono l'informazione più utile che c'è: un controllo che scatta ogni volta è un difetto del prompt, e senza il log non si vede.

Misurato: 76–184 s, mediana ~140.

In [ ]:
import logging
import time

from panel.devising import RefusedByTheChecks, devise_experience
from shared.errors import SafetyBlocked
from shared.experience import ExperienceError

logging.basicConfig(level=logging.WARNING, force=True)
logging.getLogger("panel.devising").setLevel(logging.INFO)

built: dict[str, str] = {}
began = time.time()
row: dict[str, object] = {"prompt": bench.reload_prompts(), "casa": house.name}
try:
    experience, spent = await devise_experience(**args, built_from=built, now=began)
    document = experience.to_dict()
    row |= {"titolo": experience.title, "momenti": len(experience.moments)}
except (RefusedByTheChecks, ExperienceError, SafetyBlocked) as exc:
    experience = document = None
    row["rifiutato"] = f"{type(exc).__name__}: {exc}"
row |= {"secondi": round(time.time() - began, 1), **built}
TRIED.append(row)
row

In [ ]:
from tools.as_it_arrives import read

counted = read(document)

## 5. I controlli

Qui saranno quasi sempre vuoti: `devise_experience` ripara una volta prima di restituire. I reclami che ci sono stati stanno nelle righe di log della cella sopra, ed è quella la cosa da guardare — un reclamo che torna in ogni pomeriggio nomina la regola del prompt su cui lavorare.

In [ ]:
from shared.experience_checks import check

complaints = check(experience, recent=args["recent"], sheets_at_most=args["sheets"])
print("; ".join(map(str, complaints)) or "nessun reclamo")

## 6. Giocarlo con nessuno nella stanza

`research/play.py` cammina i momenti come fa la casa e chiede a un modello, al posto della persona, che cosa è tornato sul foglio. Due cose che non fa, e sono dichiarate: nessuna pagina viene disegnata (il foglio arriva come le parole che ci sarebbero stampate sopra) e un ramo `ask` chiude la corsa invece di comprare una continuazione.

L'umore è l'unica manopola, ed è una proprietà della giornata: serve a raggiungere il ramo del foglio che torna bianco, che una simulazione volenterosa non tocca mai.

In [ ]:
from research.calls import a_context
from research.play import play
from shared.experience import Weight

ctx = a_context(time.time())
played = await play(
    ctx,
    experience=experience,
    household=house.name,
    weight=Weight.STANDARD,
    mood="una giornata normale, c'è voglia di fare qualcosa",
)
print(f"finito come: {played.ending} · {played.minutes} min · fermo a {played.reached}\n")
print(played.transcript())

## 7. Gli otto assi, su questo pomeriggio solo

Ogni asse pretende una riga citata alla lettera. È quella la parte utile: un punteggio senza citazione non dice quale frase del prompt cambiare. Un pomeriggio solo non dice se un prompt è migliorato — dice che cosa guardare.

In [ ]:
from research.calls import appraise

scored = await appraise(ctx, transcript=played.transcript())
for axis, said in (scored.get("axes") or {}).items():
    print(f"{said.get('score')}  {axis:26} {said.get('says', '')}")
print("\nal prompt:", scored.get("whatToChangeInThePrompt", "—"))

## 8. Il giro

Si apre il `.md` nell'editor, si cambia una frase, si salva, si esegue la cella qui sotto e si torna alla cella 4. L'impronta cambia solo se il testo che si manda è cambiato davvero: due pomeriggi sotto la stessa impronta sono stati scritti dallo stesso prompt, e due sotto impronte diverse non si confrontano per quanto si somiglino.

`bench.write` esiste per cambiare un blocco da qui quando è più comodo, ma l'editor va benissimo: la cella legge dal disco in ogni caso.

In [ ]:
print("impronta:", bench.reload_prompts())

for one in TRIED:
    print(one)

## 9. Quando un blocco è deciso

Non c'è niente da ricopiare: il file che il banco ha letto è quello che l'applicazione manda. Restano tre passi, in quest'ordine.

```
python -m tools.prompts --write          # rende docs/prompts/, altrimenti un test fallisce
python -m pytest -q                      # l'impronta è coperta da un test suo
python -m research.run --iterations 4 --seed 0 --label <nome>
```

L'ultimo è la misura: sei case × quattro iterazioni, circa un'ora e ~0,36 € di token, e scrive l'impronta del prompt nel sommario. È la sola cosa che risponde alla domanda «è migliorato».

⚠️ Non far girare `pytest` e una corsa insieme: `tests/test_trail.py` fallisce con `Event loop is closed` per contesa, e sembra una regressione vera.

⚠️ L'hub in casa si sincronizza a mano. Un vincolo cambiato in `shared/` vale nel cloud e non in casa finché non si copia: `powershell -NoProfile -File scripts\hub-stale.ps1`.